# gradient-routing-slm: full run on a Kaggle T4
Settings: **Accelerator = GPU T4** (x1 or x2, only one is used), **Internet = On**. Then Run All.
Results land in `/kaggle/working/out/` (plus `run.log`).

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/gradient-routing-slm.git"  # <-- set this

In [ ]:
!pip install -q -U tiktoken datasets

In [ ]:
!rm -rf /tmp/repo && git clone --depth 1 {REPO_URL} /tmp/repo
%cd /tmp/repo
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import subprocess, shutil, os, json
rc = None
try:
    p = subprocess.Popen(["bash", "run_all.sh"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with open("run.log", "w") as log:
        for line in p.stdout:
            print(line, end="")
            log.write(line)
    rc = p.wait()
finally:
    os.makedirs("/kaggle/working/out", exist_ok=True)
    if os.path.isdir("out"):
        shutil.copytree("out", "/kaggle/working/out", dirs_exist_ok=True)
    if os.path.exists("run.log"):
        shutil.copy("run.log", "/kaggle/working/run.log")
    if os.path.exists("data/out/meta.json"):
        shutil.copy("data/out/meta.json", "/kaggle/working/out/data_meta.json")
assert rc == 0, f"run_all.sh failed with exit code {rc} - see run.log"
mode = json.load(open("data/out/meta.json"))["mode"]
print()
print("DATA MODE:", mode, "" if mode == "real" else " <-- WARNING: HF download failed, results are on synthetic data!")
print(os.listdir("/kaggle/working/out"))

In [ ]:
from IPython.display import Image, display
display(Image("/kaggle/working/out/loss_curves.png"))
display(Image("/kaggle/working/out/ablation_heatmap.png"))
print(open("/kaggle/working/out/ablation_summary.txt").read())
print(open("/kaggle/working/out/samples.txt").read())